In [23]:
#loading the data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
print('Libraries imported successfully')  

dirty_data = pd.read_csv('dirty_retail_store_sales.csv')
dirty_data.head()

Libraries imported successfully


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [71]:
#INITIAL STRUCTURAL AUDIT
# checking the structural issues in the data

print("Data structure:")
dirty_data.info()
print('shape of the data:', dirty_data.shape)
dirty_data.describe()
dirty_data['Transaction Date'] = pd.to_datetime(dirty_data['Transaction Date'])

# Removes spaces at the very beginning and very end of strings
dirty_data['Category'] = dirty_data['Category'].str.strip()
dirty_data['Discount Applied'].unique()
      

Data structure:
<class 'pandas.DataFrame'>
Index: 11971 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    11971 non-null  str    
 1   Customer ID       11971 non-null  str    
 2   Category          11971 non-null  str    
 3   Item              11971 non-null  str    
 4   Price Per Unit    11971 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    11971 non-null  str    
 8   Location          11971 non-null  str    
 9   Transaction Date  11971 non-null  str    
 10  Discount Applied  11971 non-null  object 
dtypes: float64(3), object(1), str(7)
memory usage: 1.1+ MB
shape of the data: (11971, 11)


array([True, False], dtype=object)

In [35]:
#handling missing values
dirty_data.isnull().sum()
#dirty_data['Total Spent']=dirty_data['Price Per Unit']*dirty_data['Quantity']
# Group by Product Name or Category and fill missing prices with the most frequent price (mode)
dirty_data['Price Per Unit'] = dirty_data['Price Per Unit'].fillna(
    dirty_data.groupby('Item')['Price Per Unit'].transform(lambda x: x.mode()[0] if not x.mode().empty else np.nan)
)

# Fallback: If a product is completely new, fill with its Category's median price
dirty_data['Price Per Unit'] = dirty_data['Price Per Unit'].fillna(
    dirty_data.groupby('Category')['Price Per Unit'].transform('median')
)
# If we have Total Spent and Price Per Unit, calculate Quantity
# We round to the nearest whole number because retail quantities are discrete integers
mask_qty = dirty_data['Quantity'].isna() & dirty_data['Total Spent'].notna() & (dirty_data['Price Per Unit'] > 0)
dirty_data.loc[mask_qty, 'Quantity'] = (dirty_data.loc[mask_qty, 'Total Spent'] / dirty_data.loc[mask_qty, 'Price Per Unit']).round()

mask_total = dirty_data['Total Spent'].isna() & dirty_data['Price Per Unit'].notna() & dirty_data['Quantity'].notna()
dirty_data.loc[mask_total, 'Total Spent'] = dirty_data.loc[mask_total, 'Price Per Unit'] * dirty_data.loc[mask_total, 'Quantity']

# Handle missing item names using Category placeholders
dirty_data['Item'] = dirty_data['Item'].fillna('Unknown ' + dirty_data['Category'].astype(str) + ' Item') 

# Treat missing discounts as 0% or 0 amount
dirty_data['Discount Applied'] = dirty_data['Discount Applied'].fillna(0)

# Drop rows where both Quantity and Total Spent are missing
dirty_data = dirty_data.dropna(subset=['Quantity', 'Total Spent'], how='all')

In [ ]:

dirty_data.isna().sum()
dirty_data.duplicated().sum()



In [65]:
#LOGICAL INTEGRITY CHECKS

dirty_data.duplicated().sum()
(dirty_data['Total Spent'] <= 0).sum()
(dirty_data['Quantity'] <= 0).sum()
(dirty_data['Price Per Unit'] <= 0).sum()

Q1 = dirty_data['Price Per Unit'].quantile(0.25)
Q3 = dirty_data['Price Per Unit'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

non_normal_sales = dirty_data[(dirty_data['Price Per Unit'] < lower_bound) | (dirty_data['Price Per Unit'] > upper_bound)]
print(f"Number of non-normal sales: {len(non_normal_sales)}")

Q1 = dirty_data['Quantity'].quantile(0.25)
Q3 = dirty_data['Quantity'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

non_normal_sales = dirty_data[(dirty_data['Quantity'] < lower_bound) | (dirty_data['Quantity'] > upper_bound)]
print(f"Number of non-normal sales: {len(non_normal_sales)}")



Number of non-normal sales: 0
Number of non-normal sales: 0


In [66]:
# Keep only rows where price and quantity are strictly positive
#print(  dirty_data[(dirty_data['Price Per Unit'] > 0) & (dirty_data['Quantity'] > 0)])

dirty_data.duplicated().sum()

np.int64(0)

In [ ]:
# ADVANCDED FEATURE ENGINEERING
dirty_data['Day_of_Week'] = dirty_data['Transaction Date'].dt.dayofweek

dirty_data['Is_Weekend'] = dirty_data['Transaction Date'].dt.dayofweek.isin([5, 6]).astype(int)
dirty_data.head()

In [76]:
weekend_analysis = dirty_data.groupby(['Category', 'Is_Weekend'])['Total Spent'].mean().unstack()
print(weekend_analysis)

Is_Weekend                                   0           1
Category                                                  
Beverages                           130.177285  136.572639
Butchers                            137.554782  144.418854
Computers and electric accessories  126.584746  136.360241
Electric household essentials       136.678199  127.891540
Food                                130.256872  127.986726
Furniture                           126.830841  130.846154
Milk Products                       120.134011  117.024362
Patisserie                          126.880019  125.210127
